# M7 — quick PROBE (does the CNN learn anything?)

**Not the final M7.** Test: does a small 1D-CNN on the raw signal show any WPW signal (no-GPU, 142 WPW)? A single train(folds 1-8)/test(fold 9) split, subsampled negatives, signal downsampled to 1024, a tiny network, multi-seed. **fold10 NEVER touched.** The real M7 (full OOF) comes later.

> Prerequisite: **PyTorch** (CPU). If missing: `pip install torch --index-url https://download.pytorch.org/whl/cpu`


### p0 : Setup + config probe

In [ ]:
# M7 PROBE (quick, NON-rigorous) — just: does a small 1D-CNN learn anything about WPW?
# Assumed shortcuts (this is NOT the final M7): a single train(folds 1-8)/test(fold 9) split, negatives
# subsampled (speed + imbalance), downsampled signal, tiny network, CPU. fold10 NEVER touched.
# The real M7 (full OOF template, careful augmentation, optional pretraining) comes later.
import os, sys, json, time, warnings
import numpy as np, pandas as pd
from scipy.signal import butter, sosfiltfilt, resample
from sklearn.metrics import average_precision_score, roc_auc_score
from joblib import Parallel, delayed
from tqdm import tqdm
warnings.filterwarnings('ignore')
ROOT=r'.'
PROCESSED=os.path.join(ROOT,'data','processed'); SRC=os.path.join(ROOT,'src')
sys.path.insert(0,SRC); from signal_loading import load_signal, LEADS_CANONICAL
with open(os.path.join(PROCESSED,'filter_config.json'),encoding='utf-8') as f: FCFG=json.load(f)['filter_FINAL']
FS=FCFG['fs']; LEADS=list(LEADS_CANONICAL); LEAD_IDX={L:LEADS_CANONICAL.index(L) for L in LEADS}
SOS=butter(FCFG['order'],[FCFG['low']/(FS/2),FCFG['high']/(FS/2)],btype='band',output='sos')
def bp(x): return sosfiltfilt(SOS,np.asarray(x,dtype=np.float64))
# ---- probe config ----
L=1024              # resampled signal (12 x L)
N_NEG_TRAIN=8000    # subsample of non-WPW for training
EPOCHS=40; PATIENCE=8; BATCH=64; SEEDS=[0,1,2]; LR=1e-3
meta=pd.read_csv(os.path.join(PROCESSED,'metadata_combined.csv'),dtype={'ecg_id':str})
print(f"M7 PROBE | resample L={L} | neg train subsample {N_NEG_TRAIN} | epochs {EPOCHS} | seeds {SEEDS}")
print(f"WPW: train(1-8) {int(meta[(meta.fold.between(1,8))&(meta.label==1)].shape[0])} | fold9 {int(meta[(meta.fold==9)&(meta.label==1)].shape[0])} [test] | fold10 {int(meta[(meta.fold==10)&(meta.label==1)].shape[0])} [UNTOUCHED]")
print("Block p0 OK.")


### p1: Data (load -> filter -> resample -> z-score), cached

In [ ]:
# Data: load -> bandpass -> resample to (12,L) -> per-lead z-score. Train = folds 1-8 (ALL 115 WPW + N_NEG_TRAIN
# subsampled non-WPW). Test = fold 9 (full, realistic prevalence). Cached to .npz (one-time I/O).
def load_ecg(eid,source):
    raw=load_signal(eid,source); x=np.stack([bp(raw[:,LEAD_IDX[Ld]]) for Ld in LEADS])   # (12,T)
    x=resample(x,L,axis=1); x=(x-x.mean(1,keepdims=True))/(x.std(1,keepdims=True)+1e-6)
    return x.astype('float32')
CACHE=os.path.join(PROCESSED,'m7probe_data.npz')
if os.path.exists(CACHE):
    z=np.load(CACHE); Xtr,ytr,Xte,yte=z['Xtr'],z['ytr'],z['Xte'],z['yte']; print(f"[guard] cache reloaded")
else:
    tr=meta[meta.fold.between(1,8)]; wpw=tr[tr.label==1]; neg=tr[tr.label==0].sample(min(N_NEG_TRAIN,(tr.label==0).sum()),random_state=0)
    trm=pd.concat([wpw,neg]).sample(frac=1,random_state=0).reset_index(drop=True)
    tem=meta[meta.fold==9].reset_index(drop=True)
    print(f"loading train {len(trm)} ({int((trm.label==1).sum())} WPW) + test {len(tem)} ({int((tem.label==1).sum())} WPW)...")
    Xtr=np.stack(Parallel(n_jobs=6)(delayed(load_ecg)(r.ecg_id,r.source) for _,r in tqdm(trm.iterrows(),total=len(trm),desc='train')))
    Xte=np.stack(Parallel(n_jobs=6)(delayed(load_ecg)(r.ecg_id,r.source) for _,r in tqdm(tem.iterrows(),total=len(tem),desc='test')))
    ytr=trm.label.values.astype('float32'); yte=tem.label.values.astype('float32')
    np.savez_compressed(CACHE,Xtr=Xtr,ytr=ytr,Xte=Xte,yte=yte)
print(f"Xtr {Xtr.shape} ({int(ytr.sum())} WPW) | Xte {Xte.shape} ({int(yte.sum())} WPW)")
Xtr=np.nan_to_num(Xtr).astype('float32'); Xte=np.nan_to_num(Xte).astype('float32')


### p2: Small 1D-CNN + training (weighted BCE, augmentation, early stop, grad-clip) + multi-seed + fold9 eval

In [ ]:
# Tiny 1D-CNN + train (weighted BCE for imbalance, light augmentation, early stopping on an internal split) +
# multi-seed (mean of probabilities) + fold9 eval. Vs references M3 0.619 / M4 0.718 (OOF AP).
import torch, torch.nn as nn
torch.set_num_threads(6)
class TinyCNN(nn.Module):
    def __init__(s):
        super().__init__()
        s.net=nn.Sequential(
            nn.Conv1d(12,16,7,padding=3), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(16,32,5,padding=2), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(4),
            nn.Conv1d(32,32,3,padding=1), nn.BatchNorm1d(32), nn.ReLU(), nn.AdaptiveMaxPool1d(1))
        s.head=nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(32,1))
    def forward(s,x): return s.head(s.net(x)).squeeze(1)
def augment(xb):
    sh=int(torch.randint(-40,41,(1,)).item()); xb=torch.roll(xb,sh,dims=2)
    xb=xb*(0.8+0.4*torch.rand(xb.shape[0],1,1)); xb=xb+0.02*torch.randn_like(xb); return xb
def predict(model,X):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(X),256): out.append(torch.sigmoid(model(torch.tensor(X[i:i+256]))).numpy())
    return np.nan_to_num(np.concatenate(out))
def train_eval(seed):
    torch.manual_seed(seed); rng=np.random.default_rng(seed)
    idx=np.arange(len(ytr)); pos=idx[ytr==1].copy(); neg=idx[ytr==0].copy(); rng.shuffle(pos); rng.shuffle(neg)
    nvp=max(8,int(0.15*len(pos))); nvn=int(0.15*len(neg))
    vidx=np.concatenate([pos[:nvp],neg[:nvn]]); tidx=np.concatenate([pos[nvp:],neg[nvn:]])
    Xt=torch.tensor(Xtr[tidx]); yt=torch.tensor(ytr[tidx]); Xv=Xtr[vidx]; yv=ytr[vidx]
    model=TinyCNN(); opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=1e-4)
    pw=torch.tensor([float((yt==0).sum())/max(float((yt==1).sum()),1.0)])
    lossf=nn.BCEWithLogitsLoss(pos_weight=pw)
    best=-1.0; best_state=None; bad=0
    for ep in range(EPOCHS):
        model.train(); perm=torch.randperm(len(Xt))
        for i in range(0,len(Xt),BATCH):
            b=perm[i:i+BATCH]
            if len(b)<2: continue                      # BatchNorm needs >=2 (size-1 batch -> NaN)
            xb=augment(Xt[b].clone())
            opt.zero_grad(); loss=lossf(model(xb),yt[b]); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0); opt.step()
        ap=average_precision_score(yv,predict(model,Xv))
        if ap>best: best=ap; best_state={k:v.clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=PATIENCE: break
    model.load_state_dict(best_state)
    return predict(model,Xte), best, ep+1
probs=[]; 
for s in SEEDS:
    t0=time.time(); p,vap,ne=train_eval(s); probs.append(p)
    print(f"  seed {s}: best val AP {vap:.3f} | {ne} epochs | {(time.time()-t0)/60:.1f} min | fold9 AP {average_precision_score(yte,p):.3f}")
P=np.mean(probs,axis=0)
ap=average_precision_score(yte,P); auc=roc_auc_score(yte,P)
print(f"\n=== M7 PROBE — fold9 (test, {int(yte.sum())} WPW) ===")
print(f"  ensemble({len(SEEDS)} seeds) AP {ap:.3f} | AUC {auc:.3f}")
print(f"  REF (AP OOF): M3 0.619 | M4 0.718 | M3+M4 vote 0.736")
print(f"  -> reading: if AP >~ 0.4-0.5, the CNN captures a real signal and M7 is worth pursuing fully;")
print(f"     if AP ~ prevalence/noise, deep learning struggles here (data-limited) — useful to know.")
print("  NOTE: non-rigorous probe (1 split, subsampled negatives, noisy fold9=13 WPW). The real M7 will do the full OOF.")
